In [6]:
import pandas as pd

In [7]:
#Import du df -> adresse du csv à vérifier à récupérer sur votre ordinateur
adresse=r"C:\Users\Admin\Documents\_Documents\@ Fac\Anciens cours de Fac\2024-2025 Projets\Hackathon\Projet2\data\données_traitées\Bibliographie_de_la_France_Tables_Littérature_Version détaillée_1824_v3.csv" #csv annoté (1824_v3) avec prédictions
df=pd.read_csv(adresse, encoding="utf-8",sep=";",header=0,dtype=str)
df = df.fillna("")
df['Identifiant ouvrage']=df['Identifiant ouvrage'].str.replace("http://ark.bnf.fr/ark:/","https://catalogue.bnf.fr/ark:/")
df["Identifiant auteur"]=df["Identifiant auteur"].str.replace("http://ark.bnf.fr/ark:/","https://catalogue.bnf.fr/ark:/")
df["Identifiant para-auteur"]=df["Identifiant para-auteur"].str.replace("http://ark.bnf.fr/ark:/","https://catalogue.bnf.fr/ark:/")

In [ ]:
def verification_ouvrage(row):
    verite=row['Identifiant ouvrage'].split(";")
    prediction=row['Ouvrage_prédit'].split(" , ")
    resultat="Hors BNF"
    for vrai_ouvrage in verite:
        if not vrai_ouvrage.startswith("https://catalogue.bnf.fr/ark:/"):#Il ne s'agit pas d'un lien BNF, l'algorithme n'est normalement pas en capacité de le trouver
            continue 
        elif vrai_ouvrage in prediction:
            resultat="Prediction correcte"
            break #Dans le cas de multiples ouvrages, on se contentera d'un seul correctement prédit
        elif prediction==[""]:
            resultat="Non trouvé"
        else:
            resultat="Pas le même ouvrage"
    return resultat
df['predict_ouvrage_verif']=df.apply(verification_ouvrage,axis=1)
print(df['predict_ouvrage_verif'].value_counts())

predict_ouvrage_verif
Prediction correcte    1304
Non trouvé              384
Hors BNF                232
Pas le même ouvrage     187
Name: count, dtype: int64


In [9]:
#Visualisation direct des résultats d'un type (ouvrage)
verif_ouvrage="Non trouvé" #En choisir un parmi "Prediction correcte", "Non trouvé", "Hors BNF", "Pas le même ouvrage"
df[df['predict_ouvrage_verif']==verif_ouvrage][["Titre","Auteur","Identifiant ouvrage","Ouvrage_prédit","Auteur_prédit",'predict_ouvrage_verif']]

,Titre,Auteur,Identifiant ouvrage,Ouvrage_prédit,Auteur_prédit,predict_ouvrage_verif
11,Réflexions rapides sur une rébellion de débite...,l'Hermite du quartier Notre-Dame,https://catalogue.bnf.fr/ark:/12148/cb36388010m,,,Non trouvé
16,L’Entrée de Ferdinand VII à Madrid ; stances p...,"J. Brisset, garde du corps du roi de France",https://catalogue.bnf.fr/ark:/12148/cb301627424,,,Non trouvé
21,"Le Tambour de Logrono, ou Jeunesse et Valeur, ...",Adolphe C***;Combault,https://catalogue.bnf.fr/ark:/12148/cb30520618c,,,Non trouvé
22,"L’Arc de triomphe, tableau-vaudeville […], rep...",M. Carmouche;M. Emile Vander Burch,https://catalogue.bnf.fr/ark:/12148/cb301974366,,,Non trouvé
24,"Mon hommage à S. A. R. Mgr le duc d’Angoulême,...",Capelle,https://catalogue.bnf.fr/ark:/12148/cb30193716f,,,Non trouvé
...,...,...,...,...,...,...
2068,Plainte adressée à MM. les membres de l'Académ...,"P. O. Mariavel, professeur de langue anglaise",https://catalogue.bnf.fr/ark:/12148/cb30883259b,,,Non trouvé
2071,"La Mort de Henri IV, poëme en dix chants.","Paillet de Plombières, membre de l'Athénée des...",https://catalogue.bnf.fr/ark:/12148/cb31050596s,,,Non trouvé
2078,"L’Intrigante, satire",,https://catalogue.bnf.fr/ark:/12148/cb33437854x,,,Non trouvé
2092,"Le Plâtrier, ou la double accusation, mélodram...",Saint-Amand;Jules;Henri,https://catalogue.bnf.fr/ark:/12148/cb30709345j,,,Non trouvé


In [ ]:
def verification_auteur_simplifie(row):
    auteur_True,paraauteur_True,auteur_predict,paraauteur_predict=row["Identifiant auteur"],row["Identifiant para-auteur"],row["Auteur_prédit"],row["Autre_auteur_prédit"]
    auteur_True=auteur_True.split(";") if auteur_True else []
    auteur_True=set([code for code in auteur_True if "https://catalogue.bnf.fr/" in code])
    paraauteur_True=paraauteur_True.split(";") if paraauteur_True else []
    paraauteur_True=set([code for code in paraauteur_True if "https://catalogue.bnf.fr/" in code])
    auteur_predict=auteur_predict.split(";") if auteur_predict else []
    auteur_predict=set([name[name.find("http"):-2] for name in auteur_predict])
    paraauteur_predict=paraauteur_predict.split(";") if paraauteur_predict else []
    paraauteur_predict=set([name[name.find("http"):-2] for name in paraauteur_predict])
    results=dict()
    if auteur_True==auteur_predict:
        results["Auteur"]="ok"
    else:
        result=""
        for auteur in auteur_True:
            if auteur in auteur_predict:
                pass
            elif auteur in paraauteur_predict :
                if result!="Auteur absent":
                    result="Auteur placé en para-auteur"
            else:
                result="Auteur absent" 
                break
        if result=="":
            result="Auteur ajouté"
        results["Auteur"]=result


    if paraauteur_True==paraauteur_predict:
        results["Para-auteur"]="ok"
    else:
        result=""
        for paraauteur in paraauteur_True:
            if paraauteur in paraauteur_predict:
                pass
            elif paraauteur in auteur_predict :
                if result!="Para-auteur absent":
                    result="Para-auteur placé en auteur"
            else:
                result="Para-auteur absent" 
                break
        if result=="" or result=="Para-auteur placé en auteur":
            result="Para-auteur ajouté"
        results["Para-auteur"]=result
    return pd.Series(results)
df[['predict_auteur_verif','predict_paraauteur_verif']]=df.apply(verification_auteur_simplifie,axis=1,result_type="expand")

#Modification des résultats pours quelques auteurs correctement identifiés mais dont les données d'entraînement donne un mauvais code (un code plus valable ?)
df.loc[(df["Auteur"] == "J. J. Rousseau") & (df["Auteur_prédit"] == "Rousseau, Jean-Jacques (1712-1778) [https://catalogue.bnf.fr/ark:/12148/cb119228797]"), "auteur_verification"] = "ok"
df.loc[(df["Auteur"] == "Gresset") & (df["Auteur_prédit"] == "Gresset, Jean-Baptiste-Louis (1709-1777) [https://catalogue.bnf.fr/ark:/12148/cb12063085m]"), "auteur_verification"] = "ok"
df.loc[(df["Auteur"] == "J. Racine") & (df["Auteur_prédit"] == "Racine, Jean (1639-1699) [https://catalogue.bnf.fr/ark:/12148/cb120076761]"), "auteur_verification"] = "ok"
df.loc[(df["Auteur"] == "Ovide") & (df["Auteur_prédit"] == "Ovide (0043 av. J.-C.-0017) [https://catalogue.bnf.fr/ark:/12148/cb119183166]"), "auteur_verification"] = "ok"

print(f"--- Sur tous les textes trouvés : {len(df[df['predict_ouvrage_verif']!='Non trouvé'])} textes")
print(df[df['predict_ouvrage_verif']!="Non trouvé"]['predict_auteur_verif'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']!="Non trouvé"]['predict_paraauteur_verif'].value_counts())
print(" ")
verif_ouvrage="Prediction correcte"
print(f"--- Sur les textes correctement identifiés : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif'].value_counts())
print(" ")
verif_ouvrage="Hors BNF"
print(f"--- Sur les textes hors BNF (potentiellement identifiés quand même) : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif'].value_counts())
print(" ")
verif_ouvrage="Pas le même ouvrage"
print(f"--- Sur les textes liés à un autre de la BNF que celui prévu : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif'].value_counts())

--- Sur tous les textes trouvés : 1723 textes
predict_auteur_verif
ok                             1286
Auteur ajouté                   209
Auteur absent                   197
Auteur placé en para-auteur      31
Name: count, dtype: int64
 
predict_paraauteur_verif
ok                    1370
Para-auteur ajouté     237
Para-auteur absent     116
Name: count, dtype: int64
 
--- Sur les textes correctement identifiés : 1304 textes
predict_auteur_verif
ok                             1068
Auteur ajouté                   133
Auteur absent                    80
Auteur placé en para-auteur      23
Name: count, dtype: int64
 
predict_paraauteur_verif
ok                    1056
Para-auteur ajouté     162
Para-auteur absent      86
Name: count, dtype: int64
 
--- Sur les textes hors BNF (potentiellement identifiés quand même) : 232 textes
predict_auteur_verif
ok                             108
Auteur absent                   66
Auteur ajouté                   57
Auteur placé en para-auteur      1
N

In [11]:
#Visualisation direct des résultats d'un type (auteur)
verif_ouvrage="Hors BNF" #En choisir un parmi "Prediction correcte", "Non trouvé", "Hors BNF", "Pas le même ouvrage"
verif_auteur="ok!"  #En choisir un parmi "ok!","ok(vide)","Auteur ajouté","Auteur absent","Auteur présent","Auteur placé en para-auteur"
df[(df['predict_ouvrage_verif']==verif_ouvrage) & (df['predict_auteur_verif'].str.contains(verif_auteur))][["Titre","Auteur","Identifiant auteur","Identifiant para-auteur","Auteur_prédit","Autre_auteur_prédit",'predict_auteur_verif','predict_paraauteur_verif']]

,Titre,Auteur,Identifiant auteur,Identifiant para-auteur,Auteur_prédit,Autre_auteur_prédit,predict_auteur_verif,predict_paraauteur_verif


In [ ]:
def verification_auteur_detaille(row):
    auteur_True,paraauteur_True,auteur_predict,paraauteur_predict=row["Identifiant auteur"],row["Identifiant para-auteur"],row["Auteur_prédit"],row["Autre_auteur_prédit"]
    auteur_True=auteur_True.split(";") if auteur_True else []
    auteur_True=set([code for code in auteur_True if "https://catalogue.bnf.fr/" in code])
    paraauteur_True=paraauteur_True.split(";") if paraauteur_True else []
    paraauteur_True=set([code for code in paraauteur_True if "https://catalogue.bnf.fr/" in code])
    auteur_predict=auteur_predict.split(";") if auteur_predict else []
    auteur_predict=set([name[name.find("http"):-2] for name in auteur_predict])
    paraauteur_predict=paraauteur_predict.split(";") if paraauteur_predict else []
    paraauteur_predict=set([name[name.find("http"):-2] for name in paraauteur_predict])
    results=dict()
    if auteur_True==auteur_predict:
        if auteur_True:
            results["Auteur"]="ok!"
        else:
            results["Auteur"]="ok(vide)"
    else:
        result={"Auteur présent":0,"Auteur placé en para-auteur":0,"Auteur absent":0,"Auteur ajouté":0}
        for auteur in auteur_True:
            if auteur in auteur_predict:
                result["Auteur présent"]+=1
            elif auteur in paraauteur_predict:
                result["Auteur placé en para-auteur"]+=1
            else:
                result["Auteur absent"]+=1
        for auteur in auteur_predict:
            if auteur not in auteur_True and auteur not in paraauteur_True:
                result["Auteur ajouté"]+=1
        listtemp=[]
        for k,v in result.items():
            if v>0:
                listtemp.append(k)
        if not listtemp:
            listtemp=["ok(vide)"]
        elif listtemp==["Auteur présent"]:
            listtemp=["ok!"]
        results['Auteur']=" ; ".join(listtemp)

    if paraauteur_True==paraauteur_predict:
        if paraauteur_True:
            results["Para-auteur"]="ok!"
        else:
            results["Para-auteur"]="ok(vide)"
    else:
        result={"Para-auteur présent":0,"Para-auteur placé en auteur":0,"Para-auteur absent":0,"Para-auteur ajouté":0}
        for paraauteur in paraauteur_True:
            if paraauteur in paraauteur_predict:
                result["Para-auteur présent"]+=1
            elif paraauteur in auteur_predict:
                result["Para-auteur placé en auteur"]+=1
            else:
                result["Para-auteur absent"]+=1
        for paraauteur in paraauteur_predict:
            if paraauteur not in paraauteur_True:
                result["Para-auteur ajouté"]+=1
        listtemp=[]
        for k,v in result.items():
            if v>0:
                listtemp.append(k)
        if not listtemp:
            listtemp=["ok(vide)"]
        elif listtemp==["Para-auteur présent"]:
            listtemp=["ok!"]
        results['Para-auteur']=" ; ".join(listtemp)
    return pd.Series(results)
df[['predict_auteur_verif_detail','predict_paraauteur_verif_detail']]=df.apply(verification_auteur_detaille,axis=1,result_type="expand")

#Modification des résultats pours quelques auteurs correctement identifiés mais dont les données d'entraînement donne un mauvais code (un code plus valable ?)
df.loc[(df["Auteur"] == "J. J. Rousseau") & (df["Auteur_prédit"] == "Rousseau, Jean-Jacques (1712-1778) [https://catalogue.bnf.fr/ark:/12148/cb119228797]"), "predict_auteur_verif_detail"] = "ok"
df.loc[(df["Auteur"] == "Gresset") & (df["Auteur_prédit"] == "Gresset, Jean-Baptiste-Louis (1709-1777) [https://catalogue.bnf.fr/ark:/12148/cb12063085m]"), "predict_auteur_verif_detail"] = "ok"
df.loc[(df["Auteur"] == "J. Racine") & (df["Auteur_prédit"] == "Racine, Jean (1639-1699) [https://catalogue.bnf.fr/ark:/12148/cb120076761]"), "predict_auteur_verif_detail"] = "ok"
df.loc[(df["Auteur"] == "Ovide") & (df["Auteur_prédit"] == "Ovide (0043 av. J.-C.-0017) [https://catalogue.bnf.fr/ark:/12148/cb119183166]"), "predict_auteur_verif_detail"] = "ok"

print(f"--- Sur tous les textes : {len(df)} textes")
print(df['predict_auteur_verif_detail'].value_counts())
print(" ")
print(df['predict_paraauteur_verif_detail'].value_counts())
print(" ")
verif_ouvrage="Prediction correcte"
print(f"--- Sur les textes correctement identifiés : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif_detail'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif_detail'].value_counts())
print(" ")
verif_ouvrage="Hors BNF"
print(f"--- Sur les textes hors BNF (potentiellement identifiés quand même) : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif_detail'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif_detail'].value_counts())
print(" ")
verif_ouvrage="Pas le même ouvrage"
print(f"--- Sur les textes liés à un autre de la BNF que celui prévu : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif_detail'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif_detail'].value_counts())

--- Sur tous les textes : 2107 textes
predict_auteur_verif
Auteur absent    1656
ok                298
Auteur ajouté     153
Name: count, dtype: int64
 
predict_paraauteur_verif
ok                    1653
Para-auteur absent     247
Para-auteur ajouté     207
Name: count, dtype: int64
 
--- Sur les textes correctement identifiés : 1304 textes
predict_auteur_verif
Auteur absent    1100
ok                123
Auteur ajouté      81
Name: count, dtype: int64
 
predict_paraauteur_verif
ok                    992
Para-auteur absent    175
Para-auteur ajouté    137
Name: count, dtype: int64
 
--- Sur les textes hors BNF (potentiellement identifiés quand même) : 232 textes
predict_auteur_verif
Auteur absent    118
ok                59
Auteur ajouté     55
Name: count, dtype: int64
 
predict_paraauteur_verif
ok                    180
Para-auteur ajouté     33
Para-auteur absent     19
Name: count, dtype: int64
 
--- Sur les textes liés à un autre de la BNF que celui prévu : 187 textes
predict_aute

In [ ]:
#Si vous êtes plus à l'aise sur logiciel : vous pouvez le re-sauvegarder avec les colonnes de verification ainsi
adresse="" #adresse de sauvegarde
df.to_csv(adresse, encoding="utf-8", sep=";", header=True, index=False)